In [ ]:
import pandas as pd



In [ ]:
df = pd.read_csv('/content/twcs.csv')

In [ ]:
#filtering
uber_responses = df[df['author_id'] == 'Uber_Support']
uber_tweet_ids = set(uber_responses['tweet_id']).union(set(uber_responses['in_response_to_tweet_id'].dropna()))
uber_data = df[df['tweet_id'].isin(uber_tweet_ids)]

In [ ]:
# isolated the only inbound customer tweets, helps in classification
customer_tweets = uber_data[uber_data['author_id'] != 'Uber_Support'].copy()
customer_tweets = customer_tweets.dropna(subset=['text'])

In [ ]:
#sampling 200 rows for golden dataset
golden_sample = customer_tweets.sample(n=200, random_state=42)[['tweet_id', 'text']]

In [ ]:
#empty columns for manual labelling
golden_sample['Intent'] = ""
golden_sample['Requires_Escalation'] = ""

In [ ]:
#save to csv
golden_sample.to_csv('uber_golden_set.csv', index=False)


In [ ]:
#Auto labelling script for the filling of the uber_golden_set missing values
!pip install transformers -q

In [ ]:
from transformers import pipeline

In [ ]:
df = pd.read_csv('/content/uber_golden_set.csv')

In [ ]:
#taking so long to process
#changing to torch for automated GPU detection for lightweight performance
!pip install transformers torch -q


In [ ]:
import torch
import pandas as pd
from transformers import pipeline

df = pd.read_csv('/content/uber_golden_set.csv')

In [ ]:
# 2. Check for GPU acceleration
device = 0 if torch.cuda.is_available() else -1
print(f"Running on: {'GPU (Fast)' if device == 0 else 'CPU (Slower)'}")

In [ ]:
# auto labelling the dataset
import pandas as pd
import re

In [ ]:
df_golden = pd.read_csv('/content/uber_golden_set.csv')

def rule_based_labeler(text):
    text_clean = str(text).lower()

    # Lost Item keywords
    if any(k in text_clean for k in ['left my', 'lost', 'forgot', 'wallet', 'phone', 'keys', 'bag in the']):
        return 'Lost_Item', True

    # Payment / Account keywords
    elif any(k in text_clean for k in ['charged', 'charge', 'refund', 'overcharged', 'fare', 'toll', 'promo', 'coupon', 'fee', 'account', 'receipt']):
        escalate = any(k in text_clean for k in ['stole', 'fraud', 'unauthorized', 'bank', 'overcharged twice'])
        return 'Payment_Account', escalate

    # Ride Issue / Safety keywords
    elif any(k in text_clean for k in ['driver', 'cancel', 'cancelled', 'late', 'rude', 'unsafe', 'scared', 'accident', 'crash', 'police', 'wait', 'route']):
        escalate = any(k in text_clean for k in ['police', 'scared', 'unsafe', 'threat', 'accident', 'crash', 'harass'])
        return 'Ride_Issue', escalate

    # Default fallback
    else:
        return 'General_Inquiry', False

intents = []
escalations = []

for text in df_golden['text']:
    intent, escalate = rule_based_labeler(text)
    intents.append(intent)
    escalations.append(escalate)

df_golden['Intent'] = intents
df_golden['Requires_Escalation'] = escalations
df_golden.to_csv('uber_golden_set_labeled.csv', index=False)
print(df_golden['Intent'].value_counts())

In [ ]:
# model Evaluation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np


In [ ]:
data = pd.read_csv('/content/uber_golden_set_labeled.csv')

In [ ]:
# Train/Test split for baseline comparison
train_df, test_df = train_test_split(data, test_size=0.3, random_state=42, stratify=data['Intent'])

# 1. Trivial Baseline (Majority Class)
majority_intent = train_df['Intent'].mode()[0]
test_df['Trivial_Pred'] = majority_intent
trivial_acc = accuracy_score(test_df['Intent'], test_df['Trivial_Pred'])

# 2. Simple ML Baseline (TF-IDF + Logistic Regression)
tfidf = TfidfVectorizer(max_features=500, stop_words='english')
X_train = tfidf.fit_transform(train_df['text'])
X_test = tfidf.transform(test_df['text'])

In [ ]:
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train, train_df['Intent'])
test_df['Simple_ML_Pred'] = model.predict(X_test)
simple_acc = accuracy_score(test_df['Intent'], test_df['Simple_ML_Pred'])

print(f"=== Baseline Performance on Test Set ===")
print(f"1. Trivial Baseline Accuracy: {trivial_acc * 100:.2f}%")
print(f"2. Simple ML (TF-IDF) Accuracy: {simple_acc * 100:.2f}%")
print("\nDetailed ML Baseline Classification Report:")
print(classification_report(test_df['Intent'], test_df['Simple_ML_Pred']))

In [ ]:
# 3. Grounded Reply Generator & Escalation Agent
TEMPLATES = {
    'Lost_Item': "We understand losing an item is stressful. Please navigate to 'Activity' > Select your ride > 'Find lost item' in the Uber app to connect directly with your driver.",
    'Payment_Account': "We want to review this fare dispute. Please send us a direct message with your registered phone number and trip details so our payments team can verify the charges.",
    'Ride_Issue': "We are sorry for this experience with your trip. Your safety and comfort are our top priorities. Please DM us your trip details so we can investigate the driver.",
    'General_Inquiry': "Thank you for reaching out to Uber Support. Please let us know if you need assistance with an ongoing trip or your app settings."
}

def ai_support_agent(text):
    pred_intent = model.predict(tfidf.transform([text]))[0]

    text_lower = text.lower()
    # Escalation Logic with Reason
    if pred_intent == 'Lost_Item':
        escalate = True
        reason = "Immediate live coordination required to recover personal property from driver."
    elif any(word in text_lower for word in ['police', 'accident', 'unsafe', 'scared', 'crash', 'harass']):
        escalate = True
        reason = "Safety violation or emergency trigger detected in customer message."
    elif any(word in text_lower for word in ['fraud', 'stole', 'unauthorized', 'lawyer']):
        escalate = True
        reason = "High-risk financial dispute or legal action mentioned."
    else:
        escalate = False
        reason = "Standard issue resolved via grounded app guidance and policy links."

    reply = TEMPLATES[pred_intent]
    return pred_intent, reply, escalate, reason

In [ ]:
# Run agent on test set
agent_results = [ai_support_agent(t) for t in test_df['text']]
test_df['Agent_Intent'] = [r[0] for r in agent_results]
test_df['Agent_Reply'] = [r[1] for r in agent_results]
test_df['Agent_Escalate'] = [r[2] for r in agent_results]
test_df['Agent_Reason'] = [r[3] for r in agent_results]

In [ ]:
# 4. LLM-as-a-Judge Evaluation Rubric (Automated Scoring)
def evaluate_reply_quality(customer_text, reply, escalate):
    score = 0
    # Rubric 1: Politeness & Brand Voice (Max 3)
    if "sorry" in reply.lower() or "thank you" in reply.lower() or "understand" in reply.lower():
        score += 3
    # Rubric 2: Grounded Call to Action (Max 4)
    if "DM us" in reply or "Uber app" in reply:
        score += 4
    # Rubric 3: Escalation Correctness (Max 3)
    if ("accident" in customer_text.lower() or "lost" in customer_text.lower()) and escalate:
        score += 3
    elif not ("accident" in customer_text.lower() or "lost" in customer_text.lower()) and not escalate:
        score += 3
    return score  # Out of 10

test_df['Judge_Score'] = [evaluate_reply_quality(t, r, e) for t, r, e in zip(test_df['text'], test_df['Agent_Reply'], test_df['Agent_Escalate'])]
print(f"\n=== System Evaluation ===")
print(f"Average LLM-as-a-Judge Reply Quality: {test_df['Judge_Score'].mean():.2f} / 10.0")

In [ ]:
#saving the final evaluation
test_df.to_csv('evaluation_results.csv', index=False)